# LLM Model Routing — Human Review

This notebook walks through the per-agent / per-task LLM routing added in Step 7. The design doc is `project_planning/LLM_MODEL_ROUTING.md`.

## Two-axis lookup

Every `(agent, task)` pair resolves to a `ModelConfig` via two axes:

- **capability** — `coding`, `balanced`, or `reasoning`. Picks the model family.
- **cost_tier** — `cheap`, `moderate`, or `expensive`. Picks the tier within that family.

`config/settings.yaml` holds a `model_matrix` (capability × cost → model name) and a `capability_settings` block (temperature / max_tokens per capability). The `routes` block maps `(agent, task)` onto `{capability, cost}` pairs.

## Resolution order

`resolve_model_config(settings, agent=..., task=...)` walks:

1. `routes[agent][task]` if present,
2. else `routes[agent]['default']`,
3. else `routes[agent]` if it's a `{capability, cost}` shorthand,
4. else `routes['default']`.

After the route is picked, `settings['llm']['cost_override']` (populated from the `LLM_COST_OVERRIDE` env var in `app.py`) can downshift the cost while preserving the capability.

## How to read the routes table

Each row below shows the `(agent, task)` pair the agent code passes, the resolved capability and cost, and the concrete model name + sampling settings the adapter will use at call time. The `profile_label` is `<capability>_<cost>` and is intended to propagate into LangSmith span metadata once Phase (e) lands.

In [ ]:
import pandas as pd

from multi_agent_ds.adapters.llm import build_adapter, resolve_model_config
from multi_agent_ds.core import load_settings

settings = load_settings()
settings['llm']['routes'].keys()

## Sanity table — every (agent, task) from the intent table

This is the full list of `(agent, task)` pairs the pipeline calls today, rendered against the live config. If a route is wrong, the capability or cost_tier column will surface it immediately.

In [ ]:
intent_pairs = [
    ('eda_analyst', 'eda_review'),
    ('eda_analyst', 'prep_plan'),
    ('eda_analyst', 'processed_approval'),
    ('ml_reviewer', 'raw_review'),
    ('ml_reviewer', 'baseline_review'),
    ('ml_reviewer', 'tuning_review'),
    ('business_stakeholder', 'raw_review'),
    ('business_stakeholder', 'report_review'),
    ('data_engineer', 'feedback'),
    ('data_engineer', 'execute'),
    ('ml_modeler', 'eda_review'),
    ('ml_modeler', 'baseline_decision'),
    ('ml_modeler', 'tuning_decision'),
    ('ml_modeler', 'learning_rate_decision'),
    ('ml_modeler', 'feature_selection_decision'),
    ('ml_modeler', 'modeling_verdict'),
    ('ml_modeler', 'modeling_handoff'),
    ('report_writer', 'generate'),
]

rows = []
for agent, task in intent_pairs:
    cfg = resolve_model_config(settings, agent=agent, task=task)
    rows.append({
        'agent': agent,
        'task': task,
        'capability': cfg.capability,
        'cost_tier': cfg.cost_tier,
        'model': cfg.model,
        'temperature': cfg.temperature,
        'max_tokens': cfg.max_tokens,
        'profile_label': cfg.profile_label,
    })

pd.DataFrame(rows)

## Cost-override demo — `LLM_COST_OVERRIDE=cheap`

When the env var forces every route onto the cheap tier, the reasoning routes switch from `o3` to `o4-mini` (capability preserved). Balanced routes don't change because they're already on cheap. Useful for smoke tests or cost-budget experiments.

In [ ]:
cheap_rows = []
for agent, task in intent_pairs:
    default_cfg = resolve_model_config(settings, agent=agent, task=task)
    cheap_cfg = resolve_model_config(settings, agent=agent, task=task, cost_override='cheap')
    changed = default_cfg.model != cheap_cfg.model
    cheap_rows.append({
        'agent': agent,
        'task': task,
        'default_model': default_cfg.model,
        'cheap_override_model': cheap_cfg.model,
        'changed': changed,
        'capability_preserved': default_cfg.capability == cheap_cfg.capability,
    })

pd.DataFrame(cheap_rows)

## Fallback chain demo

- Unknown task on a known agent falls back to `routes[agent]['default']` (or the shorthand).
- Unknown agent falls back to `routes['default']`.

In [ ]:
unknown_task = resolve_model_config(settings, agent='ml_modeler', task='this_task_does_not_exist')
unknown_agent = resolve_model_config(settings, agent='some_new_agent', task=None)

print('unknown task on known agent:', unknown_task.profile_label, '→', unknown_task.model)
print('unknown agent:             ', unknown_agent.profile_label, '→', unknown_agent.model)

## Failure demos — actionable error messages

The resolver raises `ValueError` with a uniform shape. The message names the offending key and surfaces the valid options, so a malformed `settings.yaml` surfaces immediately at the first node that tries to route through it.

In [ ]:
# (i) Unknown capability in the route entry
broken_capability = {
    **settings,
    'llm': {
        **settings['llm'],
        'routes': {
            **settings['llm']['routes'],
            'broken_agent': {'capability': 'telepathy', 'cost': 'cheap'},
        },
    },
}
try:
    resolve_model_config(broken_capability, agent='broken_agent', task=None)
except ValueError as exc:
    print('(i) unknown capability:', exc)

# (ii) Unknown cost_override value
try:
    resolve_model_config(settings, agent='eda_analyst', task=None, cost_override='bogus')
except ValueError as exc:
    print('(ii) unknown cost_override:', exc)

## What is NOT in this slice

- **Phase (e) — LangSmith metadata propagation.** The `ModelConfig` already carries `capability`, `cost_tier`, and `profile_label` but they are not yet emitted as span metadata. Deferred until `langsmith` is added as a project dependency; `routing.py` has a `TODO(phase-e)` flagging the follow-up.
- **Multi-provider adapters.** `ModelConfig.provider` is populated (currently `'openai'`) but only `OpenAIAdapter` exists. Anthropic / Google adapters are future work per `project_planning/FUTURE_WORK.md` §2.5.
- **Per-route `max_tokens` override.** Today `max_tokens` is set at the capability level. A follow-up can promote `max_tokens` into the route entry when token-overflow shows up in traces.
- **Startup config validation.** `resolve_model_config` is lazy — a malformed config only surfaces at the first offending call. A `validate_routing_config(settings)` pass at startup is on the deferred list.
- **Retry backoff bump.** Retry base stayed at 1.0s (not bumped to 2.0s). Minimal change in this slice; revisit if reasoning-model latencies trigger false-positive retries.